# Olira Python SDK — End-to-End Demo

This notebook walks through the full Olira SDK surface: authentication, patient management, event logging, batch operations, and event querying.

**Prerequisites**
- Python 3.10+
- SDK + notebook dependencies installed (see setup below)
- An Olira API key with scopes: `api:manage-patients`, `sdk:event-management`, `sdk:patient-token`

**Setup (first time)**
```bash
cd examples/
python -m venv .venv
source .venv/bin/activate       # Windows: .venv\Scripts\activate
pip install -e .
cp .env.example .env            # then fill in your OLIRA_API_KEY
```

**VS Code**
1. Open this `.ipynb` file in VS Code
2. Select the `examples/.venv` kernel when prompted
3. Edit `.env` with your API key
4. Run All Cells — `⇧⌘P` → *"Notebook: Run All Cells"*

**Targeting localhost** — defaults point at `http://localhost:8080/app-api`. Set `OLIRA_BASE_URL` in `.env` to override.

## 1. Configuration

Values are loaded from the `.env` file in this directory. Copy `.env.example` to `.env` and fill in your `OLIRA_API_KEY` — that's the only edit needed.

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv

# Load .env from the examples/ directory (works regardless of working directory)
load_dotenv(Path(__file__).parent / ".env" if "__file__" in dir() else Path(".env"))

API_KEY = os.environ["OLIRA_API_KEY"]
BASE_URL = os.getenv("OLIRA_BASE_URL", "http://localhost:8080/app-api")

print(f"BASE_URL : {BASE_URL}")
print(f"API_KEY  : {API_KEY[:12]}..." if len(API_KEY) > 12 else f"API_KEY  : {API_KEY}")

BASE_URL : http://localhost:8080/app-api
API_KEY  : olira_dev_5c...


## 2. Authentication

Instantiate the synchronous `OliraClient`. The client performs no network call on construction — it only validates that `api_key` is non-empty. The first actual request (e.g. `create_patient`) will raise `AuthError` if the key is invalid or revoked.

**API key scopes** — keys are issued with a specific set of permission scopes:
- `api:manage-patients` — create / read / update / delete patients
- `sdk:event-management` — query and delete ingested events
- `sdk:patient-token` — mint short-lived patient JWTs

In [2]:
from olira import OliraClient, OliraEnv
from olira.exceptions import AuthError

client = OliraClient(
    api_key=API_KEY,
    base_url=BASE_URL,
    environment=OliraEnv.DEVELOPMENT,
    service_name="olira-sdk-demo",
)

print("Client instantiated successfully.")

# Demonstrate AuthError with a deliberately bad key
try:
    bad_client = OliraClient(api_key="invalid_key", base_url=BASE_URL)
    bad_client.list_patients()  # This triggers the first network call
except AuthError as e:
    print(f"AuthError (expected): {e}")
except Exception as e:
    print(f"Other error (non-auth backend may be offline): {type(e).__name__}: {e}")

Client instantiated successfully.
AuthError (expected): API key rejected (HTTP 401). Check key validity and scope.


## 3. Patient Management

Patients are the central entity in Olira. Each patient has a unique ID assigned by the platform, returned on creation and used as `patient_id` in all subsequent calls.

**Important:** `patient_id` in event payloads must be your internal identifier — never an email address, phone number, or SSN. The SDK validates this client-side and raises `ValidationError` if PII is detected.

In [ ]:
# Create a patient
new_patient = client.create_patient(
    first_name="Jane",
    last_name="Demo",
    email="jane.demo@example.com",
    date_of_birth="1985-04-12T00:00:00Z",
    sex="female",
    timezone="America/New_York",
    primary_disease_site="breast",
    disease_stage="II",
)

PATIENT_ID = new_patient.id
print(f"Created patient: {PATIENT_ID}")
print(f"  Name   : {new_patient.first_name} {new_patient.last_name}")
print(f"  Status : {new_patient.status}")

In [ ]:
# Get a single patient
fetched = client.get_patient(patient_id=PATIENT_ID)
print(f"get_patient({PATIENT_ID})")
print(f"  Status              : {fetched.status}")
print(f"  Primary disease site: {fetched.primary_disease_site}")
print(f"  Disease stage       : {fetched.disease_stage}")

In [ ]:
# List patients
page = client.list_patients(limit=10)
print(f"list_patients() → total={page.total}, has_more={page.has_more}")
for p in page.patients:
    print(f"  {p.id}  {p.first_name} {p.last_name}")

In [ ]:
# Update patient — only include fields you want to change
updated = client.update_patient(
    patient_id=PATIENT_ID,
    disease_stage="III",
)
print(f"update_patient({PATIENT_ID})")
print(f"  disease_stage: {updated.disease_stage}")

## 4. Logging Events

`client.log()` enqueues an event for asynchronous background delivery. Call `flush()` to force delivery before the process exits.

Every event requires `event_type` and `patient_id`. `payload` and `trace` are optional.

### OliraTrace

`OliraTrace(object_type, object_id)` links an event to an object in your system — a conversation, an appointment, a session, etc. Use it whenever you want to group events by the object they relate to, or query them back by that object later.

In [ ]:
from olira.models import EsasItem, OliraEventType, OliraTrace

# 1. Simple event — no payload, no trace
client.log(
    event_type=OliraEventType.USER_LOGIN,
    patient_id=PATIENT_ID,
)
print("Queued: USER_LOGIN")

# 2. Event with structured payload — ESAS-r symptom report
client.log(
    event_type=OliraEventType.SYMPTOM_REPORT,
    patient_id=PATIENT_ID,
    payload={
        "instrument": "esas_r",
        "symptoms": [
            EsasItem(name="pain", score=3).model_dump(),
            EsasItem(name="fatigue", score=5).model_dump(),
            EsasItem(name="nausea", score=1).model_dump(),
        ],
    },
)
print("Queued: SYMPTOM_REPORT (instrument=esas_r)")

# 3. Events linked to a conversation via OliraTrace
#    All three share the same trace — they can be queried together later
CONVERSATION_ID = "conv_abc123"
conv_trace = OliraTrace(object_type="conversation", object_id=CONVERSATION_ID)

client.log(
    event_type=OliraEventType.CONVERSATION_COMPLETED,
    patient_id=PATIENT_ID,
    trace=conv_trace,
    payload={"duration_seconds": 142, "message_count": 8},
)
print(f"Queued: CONVERSATION_COMPLETED (trace → conversation/{CONVERSATION_ID})")

# 4. A symptom report also linked to the same conversation
client.log(
    event_type=OliraEventType.SYMPTOM_REPORT,
    patient_id=PATIENT_ID,
    trace=conv_trace,
    payload={
        "instrument": "esas_r",
        "symptoms": [
            EsasItem(name="pain", score=2).model_dump(),
        ],
    },
)
print(f"Queued: SYMPTOM_REPORT (trace → conversation/{CONVERSATION_ID})")

# 5. Event linked to an appointment (different object_type)
client.log(
    event_type=OliraEventType.FEATURE_USED,
    patient_id=PATIENT_ID,
    trace=OliraTrace(object_type="appointment", object_id="appt_xyz789"),
    payload={"feature": "symptom_tracker"},
)
print("Queued: FEATURE_USED (trace → appointment/appt_xyz789)")

client.flush()
print("\nFlushed — all events delivered.")

## 5. Batch Logging

`log_batch()` sends multiple events in a single HTTP request. This is efficient for high-volume ingestion or backfilling historical data.

Each `EventSpec` accepts an optional `idempotency_key` — if the same key is re-submitted within 24 hours, the server deduplicates it silently. Use a deterministic key (e.g. `f"{patient_id}:{event_type}:{timestamp}"`) to safely retry on network failure.

In [ ]:
from olira.models import EventSpec

batch = [
    EventSpec(
        event_type=OliraEventType.USER_LOGIN,
        patient_id=PATIENT_ID,
        idempotency_key=f"{PATIENT_ID}:login:2024-01-15T08:00:00Z",
    ),
    EventSpec(
        event_type=OliraEventType.FEATURE_USED,
        patient_id=PATIENT_ID,
        payload={"feature": "symptom_tracker", "version": "2.1.0"},
        idempotency_key=f"{PATIENT_ID}:feature_used:2024-01-15T08:01:00Z",
    ),
    EventSpec(
        event_type=OliraEventType.USER_LOGOUT,
        patient_id=PATIENT_ID,
        timestamp="2024-01-15T09:00:00Z",
        idempotency_key=f"{PATIENT_ID}:logout:2024-01-15T09:00:00Z",
    ),
]

result = client.log_batch(batch)
print(f"log_batch() → accepted={result.accepted}, failed={result.failed}")
if result.errors:
    for err in result.errors:
        print(f"  Error at index {err.index}: [{err.code}] {err.message}")

## 6. Event Querying & Management

Query and delete events stored for a patient. Requires the `sdk:event-management` scope.

All timestamp parameters are **ISO 8601 strings** (e.g. `"2024-01-15T08:00:00Z"`).

Available filters for `get_events` and `delete_events`:
- `event_type` — filter by a single event type
- `from_timestamp` / `to_timestamp` — filter by the event's own timestamp (when it occurred)
- `ingested_after` / `ingested_before` — filter by when Olira received the event
- `offset` / `limit` — pagination

In [ ]:
# Query all events for the patient
all_events = client.get_events(patient_id=PATIENT_ID, limit=50)
print(f"All events → total={all_events.total}, has_more={all_events.has_more}")
for ev in all_events.events:
    trace_str = f"  trace={ev.trace.object_type}/{ev.trace.object_id}" if ev.trace else ""
    print(f"  [{ev.event_type}]{trace_str}")

In [ ]:
# Filter by event_type
symptom_events = client.get_events(
    patient_id=PATIENT_ID,
    event_type=OliraEventType.SYMPTOM_REPORT,
)
print(f"SYMPTOM_REPORT events → total={symptom_events.total}")
for ev in symptom_events.events:
    trace_str = f" (trace → {ev.trace.object_type}/{ev.trace.object_id})" if ev.trace else ""
    print(f"  {ev.event_id}{trace_str}")

In [ ]:
from datetime import UTC, datetime, timedelta

# Filter by ingestion time — events ingested in the last 24 hours
since = (datetime.now(tz=UTC) - timedelta(hours=24)).strftime("%Y-%m-%dT%H:%M:%SZ")
recent_events = client.get_events(
    patient_id=PATIENT_ID,
    ingested_after=since,
)
print(f"Events ingested after {since} → total={recent_events.total}")

# Filter by event timestamp range (when the event occurred, not when it was ingested)
windowed = client.get_events(
    patient_id=PATIENT_ID,
    from_timestamp="2024-01-01T00:00:00Z",
    to_timestamp="2024-12-31T23:59:59Z",
)
print(f"Events with timestamp in 2024 → total={windowed.total}")

In [ ]:
# Delete by event type
delete_result = client.delete_events(
    patient_id=PATIENT_ID,
    event_type=OliraEventType.USER_LOGIN,
)
print(f"delete_events(USER_LOGIN) → deleted_count={delete_result.deleted_count}")

# Delete by timestamp range (e.g. purge backfilled test data)
delete_range = client.delete_events(
    patient_id=PATIENT_ID,
    from_timestamp="2024-01-01T00:00:00Z",
    to_timestamp="2024-12-31T23:59:59Z",
)
print(f"delete_events(2024 range) → deleted_count={delete_range.deleted_count}")

In [ ]:
# Pull all events back and group them by their trace object
from collections import defaultdict

all_events = client.get_events(patient_id=PATIENT_ID, limit=50)

groups: dict[str, list] = defaultdict(list)
untraced = []
for ev in all_events.events:
    if ev.trace:
        key = f"{ev.trace.object_type}/{ev.trace.object_id}"
        groups[key].append(ev.event_type)
    else:
        untraced.append(ev.event_type)

print("Events by trace object:")
for obj, types in groups.items():
    print(f"  {obj}:")
    for t in types:
        print(f"    - {t}")

print(f"\nUntraced events ({len(untraced)}):")
for t in untraced:
    print(f"  - {t}")

## 7. Patient Token

`get_patient_token()` mints a short-lived JWT scoped to a single patient. Use this to authenticate your mobile / web app directly against the Olira API without exposing your server-side API key.

**Typical flow:**
1. Your backend calls `get_patient_token(patient_id)` when the patient opens the app
2. The backend forwards the token to the app over a secure channel
3. The app uses the token as a Bearer token for Olira API calls
4. The token expires automatically (`expires_in` seconds) — repeat from step 1 to refresh

Requires the `sdk:patient-token` scope on your API key.

In [ ]:
token_response = client.get_patient_token(patient_id=PATIENT_ID)

# Show a safe snippet — never log the full token in production!
token_snippet = token_response.access_token[:20] + "..."
print(f"get_patient_token({PATIENT_ID})")
print(f"  token_type : {token_response.token_type}")
print(f"  expires_in : {token_response.expires_in}s")
print(f"  scopes     : {token_response.scopes}")
print(f"  token      : {token_snippet}")

## 8. Cleanup

Delete the demo patient and close the client. `delete_patient()` performs a soft-delete — the patient record is deactivated and events are retained for audit purposes unless explicitly deleted via `delete_events()` first.

In [ ]:
# Delete remaining events first (optional — only needed for hard GDPR deletion)
remaining = client.delete_events(patient_id=PATIENT_ID, event_type=OliraEventType.SYMPTOM_REPORT)
print(f"Deleted SYMPTOM_REPORT events: {remaining.deleted_count}")

# Soft-delete the patient
client.delete_patient(patient_id=PATIENT_ID)
print(f"Deleted patient {PATIENT_ID}.")

# Close the client — flushes any remaining queued events and stops background worker
client.close()
print("Client closed. Demo complete.")